In [1]:
import pandas as pd
import re

In [2]:
REGION = 'Attica'
enc = 'utf-8'

In [3]:
data = pd.read_csv(f'../../data/{REGION}/GR_{REGION}_GRID_Landcover_2001-2022.csv', encoding=enc)
data.columns = data.columns.str.lower()

In [4]:
data.head()

,x,y,2001_01_01_lc_prop1,2001_01_01_lc_prop1_assessment,2001_01_01_lc_prop2,2001_01_01_lc_prop2_assessment,2001_01_01_lc_prop3,2001_01_01_lc_prop3_assessment,2001_01_01_lc_type1,2001_01_01_lc_type2,...,2022_01_01_lc_prop2_assessment,2022_01_01_lc_prop3,2022_01_01_lc_prop3_assessment,2022_01_01_lc_type1,2022_01_01_lc_type2,2022_01_01_lc_type3,2022_01_01_lc_type4,2022_01_01_lc_type5,2022_01_01_lw,2022_01_01_qc
0,22.925358,36.229228,22,76,20,76.0,20,76,9,9,...,85.0,20,85,9,9,4,2,2,2,0
1,22.925358,36.319061,22,70,20,70.0,20,70,9,9,...,76.0,20,76,8,8,4,2,2,2,0
2,22.925358,36.337027,22,68,20,68.0,27,53,11,9,...,81.0,10,81,1,1,7,1,1,2,0
3,22.947519,36.193293,22,69,20,69.0,20,69,9,9,...,86.0,20,86,9,9,4,2,2,2,0
4,22.947519,36.211263,22,72,20,72.0,20,72,9,9,...,91.0,30,91,10,10,1,6,6,2,0


In [5]:
lc_columns = data.columns.tolist()
lc_columns = [x.split('_')[0] for x in lc_columns]

lc_columns
lc_years = set()

for i in range(0, len(lc_columns)):
    try:
        lc_years.add(int(lc_columns[i]))
    except ValueError:
        pass

lc_years = list(lc_years)

In [6]:
dates_ = re.compile(r"[0-9]{4}_[0-9]{2}_[0-9]{2}_")
landcover_datasets_per_year = []

for year in lc_years:
    frame_name = f'landcover_{year}'
    location_columns = ['x', 'y']
    filtered_columns = [col for col in data if col.startswith(f'{year}')]
    columns_to_keep = location_columns + filtered_columns
    locals()[frame_name] = data[columns_to_keep].copy()
    locals()[frame_name].insert(2, 'year', int(year))
    locals()[frame_name] = locals()[frame_name].rename(columns=lambda x: re.sub(dates_,'',x))
    landcover_datasets_per_year.append(locals()[frame_name])

In [7]:
landcover_processed = pd.concat(landcover_datasets_per_year, ignore_index=True)
landcover_processed.reset_index(drop=True, inplace=True)

In [8]:
landcover_processed

,x,y,year,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc
0,22.925358,36.229228,2001,22,76,20,76.0,20,76,9,9,4,1,1,2,0
1,22.925358,36.319061,2001,22,70,20,70.0,20,70,9,9,4,1,1,2,0
2,22.925358,36.337027,2001,22,68,20,68.0,27,53,11,9,4,1,1,2,0
3,22.947519,36.193293,2001,22,69,20,69.0,20,69,9,9,4,1,1,2,0
4,22.947519,36.211263,2001,22,72,20,72.0,20,72,9,9,4,1,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16209,24.033311,38.187599,2022,31,96,30,96.0,30,96,10,10,1,6,6,2,0
16210,24.033311,38.205560,2022,31,97,30,97.0,30,97,10,10,1,6,6,2,0
16211,24.055468,37.756399,2022,31,97,30,97.0,30,97,10,10,1,6,6,2,0
16212,24.055468,37.774360,2022,31,90,30,90.0,30,90,10,10,1,6,6,2,0


In [9]:
for year in landcover_processed['year'].sort_values().unique():
    subset = landcover_processed[landcover_processed['year'] == year]
    subset.to_csv(f'../../data/{REGION}/GR_{REGION}_Landcover_{year}.csv', index=False, encoding=enc)

In [10]:
landcover_processed.to_csv(f'../../data/{REGION}/GR_{REGION}_Landcover_2001-2024_processed.csv', index=False, encoding=enc)